# Expert Study Notes: Introduction to LLMs – Prompting & Scaling Laws

Welcome to the comprehensive study notes based on the NPTEL lecture covering LLM prompting, scaling laws, and parameter-efficient fine-tuning (PEFT). These notes are structured to bridge the gap between high-level summaries and dense transcripts, focusing strictly on exam-relevant concepts, architectures, and performance metrics.

---

## 1. The Evolution of Machine Learning Paradigms

To understand modern Large Language Models (LLMs), it is crucial to recognize how the approach to training and utilizing models has evolved:

1. **Old-School ML Paradigm (Train $\rightarrow$ Predict):** Models are trained from scratch on task-specific labeled data and then used for inference.
2. **Pre-2020 Paradigm (Pre-train $\rightarrow$ Fine-tune $\rightarrow$ Predict):** A base model is pre-trained on massive raw corpora to learn language semantics, then explicitly fine-tuned (updating all weights) on a downstream task dataset.
3. **Modern LLM Paradigm (Pre-train $\rightarrow$ Prompt $\rightarrow$ Predict):** Popularized by the GPT-3 paper (2020). The model is massively scaled during pre-training. Instead of fine-tuning weights for new tasks, users guide the frozen model using **natural language prompts** (instructions).

---

## 2. Recap of Pre-trained Architectures

LLM architectures generally fall into three categories based on their training objectives:

| Model Type | Examples | Architecture | Pre-training Objective |
| --- | --- | --- | --- |
| **Encoder-Only** | BERT | Pure Encoder | **Masked Language Modeling (MLM):** Predicts missing/masked words using bidirectional context. |
| **Decoder-Only** | GPT-1, GPT-2, GPT-3 | Pure Decoder | **Causal Language Modeling / Autoregressive:** Predicts the next word given previous words (left-to-right). |
| **Encoder-Decoder** | T5, BART | Full Transformer | **Denoising Autoregressive:** Corrupted input is fed to the encoder, and the decoder reconstructs the original clean text. |

---

## 3. The Scaling War & Emergent Properties

The industry (Meta, OpenAI, Google) is engaged in a "scaling war," increasing model parameters and dataset sizes without fundamentally changing the Transformer framework. Scaling leads to **emergent properties**—abilities (like zero-shot reasoning) that smaller models do not possess.

### Trajectory of Model Sizes

* **ELMo:** ~93 Million parameters ($\sim$ 1B tokens training data).
* **BERT (Base/Large):** 110M / 340M parameters (12 / 24 layers).
* **RoBERTa:** Trained on $\sim$ 30B tokens.
* **GPT-3 (2020):** 175 Billion parameters (96 decoder layers, 96 attention heads, 3.2M batch size, trained on 300B tokens from Common Crawl, WebText, BookCorpus, Wikipedia).
* **Megatron-Turing NLG:** 530 Billion parameters.

> **Key Exam Note on Scaling Dynamics:** As model size increases, **batch sizes increase significantly** while the **learning rate decreases**. This allows the model to process data stably over more iterations.

### The Linear vs. Exponential Rule

Empirical evaluations (e.g., on the GLUE benchmark) show that **an exponential increase in parameters yields a roughly linear increase in accuracy**. For example, jumping from GPT-2 (1.5B) to Megatron/XLNet scales parameters massively but only yields a modest linear jump in GLUE scores (e.g., from 85% to 90%).

---

## 4. In-Context Learning & Prompting Techniques

Because LLMs are few-shot learners, we use different prompting strategies without updating model weights:

1. **Zero-Shot Prompting:** Providing only the task description and the input. No examples are given.
* *Format:* `Translate English to French: Cheese -> `


2. **One-Shot Prompting:** Providing the task description, exactly one solved example, and the input.
3. **Few-Shot Prompting:** Providing the task description, multiple examples, and the input.
* *Constraint:* You cannot provide thousands of examples because of the model's **Context Window limit** (e.g., 4K or 10K tokens). If you have 1,000+ examples, you should switch to fine-tuning.



---

## 5. Anatomy of a Discrete (Hard) Prompt

A standard hard prompt consists of specific textual components:

* **Input ($X$):** The data to process (e.g., "I love this movie").
* **Template:** The wrapper around the input and output.
* **Placeholder ($Z$ or $Y$):** Where the model generates the answer.

### Types of Prompts

* **Prefix Prompt:** The placeholder is placed at the end of the input (best for autoregressive models like GPT).
* *Example:* `The movie was horrible. Overall, it was a [Z].`


* **Cloze Prompt:** The placeholder is in the middle of the text (best for bidirectional models like BERT).
* **Answer-Label Mapping:** If the model fills $[Z]$ with "boring" or "fantastic", a secondary step maps this raw string to a formal classification label (e.g., "Negative" or "Positive").

---

## 6. The Problem: Prompt Sensitivity

Models are highly sensitive to **Hard Prompts** (natural language, discrete tokens). Minor syntactic changes cause massive accuracy fluctuations.

* *Example Task:* "Where is X located?"
* `X is located in Y` $\rightarrow$ 31% accuracy.
* `In which city, country, or state is X located?` $\rightarrow$ 19% accuracy.
* `In which country is X located? Y` $\rightarrow$ 51% accuracy.



Because finding the "optimal" hard prompt is essentially guesswork, the field shifted toward **Soft Prompting** (Continuous Prompts).

---

## 7. Soft Prompting & Parameter-Efficient Fine-Tuning (PEFT)

To solve prompt sensitivity, we use continuous embeddings that the model can learn and optimize via backpropagation, without updating the massive LLM.

### A. Prompt Tuning

* **Mechanism:** Prepend a small set of trainable parameter vectors (embeddings) *only* to the input layer.
* **Process:** The LLM's weights are completely **frozen**. During a mini-training phase, the gradient loss is backpropagated to update *only* the prepended prompt embeddings.
* **Advantage:** You can learn task-specific prompts (one vector set for QA, one for Summarization). These vectors are small and easily shared across machines.

### B. Prefix Tuning

* **Mechanism:** Similar to Prompt Tuning, but the trainable embeddings are prepended to **every single layer** of the Transformer, not just the input. It introduces slightly more parameters than prompt tuning but remains highly efficient.

### C. Hyperparameters for Soft Prompting

* **Length of Prompt:** Prepending **20 to 30 tokens** is optimal. 1 token is too few; 150 tokens offer virtually zero added benefit over 20 tokens.
* **Initialization:** For smaller models, initializing the embeddings using sampled vocabulary (e.g., Word2Vec/GloVe) or class labels yields better results. For massive models (100B+ parameters), **random initialization** works just as well.

> **Crucial Exam Takeaway:** As the model size increases (scaling to billions of parameters), **the accuracy of Prompt Tuning matches the accuracy of Full Fine-Tuning**, saving immense computational resources.

---

## 8. Empirical Observations on LLM Performance

The lecture highlighted how prompting performs across different tasks compared to state-of-the-art (SOTA) supervised/fine-tuned models:

1. **Trivia QA (Open-book QA):**
* *Result:* At the 175B parameter scale, a **few-shot GPT model out-performs a fine-tuned SOTA model**.
* *Why?* The LLM saw these trivia facts during its massive pre-training on the web. It just needed the prompt to "trigger" the right parameters.


2. **Machine Translation (e.g., English to French):**
* *Result:* Highly competitive with supervised SOTA models.
* *Caveat:* This is surprising because only **7% of the pre-training data** was non-English. The scaling law holds true here—accuracy rises monotonically without a plateau.


3. **Comprehension QA (SQuAD, CoQA - Harder Tasks):**
* *Result:* Prompt-based GPT models **struggle significantly** and fail to match fine-tuned SOTA models. For highly complex reasoning, full fine-tuning remains the superior strategy if data is available.



---

## Practice Questions (NPTEL Format)

**Q1. According to the scaling laws discussed, what is the relationship between the number of parameters and model accuracy on benchmarks like GLUE?**
a) Exponential increase in parameters leads to exponential increase in accuracy.
b) Exponential increase in parameters leads to linear increase in accuracy.
c) Linear increase in parameters leads to exponential increase in accuracy.
d) Accuracy plateaus immediately after 1 Billion parameters.
*Answer: (b). The transcript explicitly notes that while parameter counts grow exponentially, the accuracy (GLUE scores) scales linearly.*

**Q2. In Prompt Tuning, which parameters are updated during the learning phase?**
a) All parameters in the encoder and decoder.
b) Only the parameters in the adapter layers.
c) Only the prepended continuous embeddings at the input.
d) The prepended embeddings at every layer of the model.
*Answer: (c). Prompt tuning updates only the prepended embeddings at the input layer, keeping the LLM frozen. (Option 'd' describes Prefix Tuning).*

**Q3. Why is "Soft Prompting" preferred over "Hard Prompting" for robust model deployment?**
a) Hard prompts require too much GPU memory.
b) Soft prompts are easily readable by humans.
c) Models exhibit extreme sensitivity and accuracy variance based on minor syntactic changes in hard prompts.
d) Hard prompts cannot be used for autoregressive models.
*Answer: (c). Minor changes (like swapping a colon for a question mark) drastically alter accuracy, making hard prompts unpredictable.*

**Q4. Based on empirical results, on which of the following tasks does a few-shot 175B parameter LLM struggle to match a fine-tuned SOTA model?**
a) Trivia QA
b) Machine Translation
c) Hard Comprehension QA (e.g., CoQA)
d) Both A and B
*Answer: (c). The lecture states that while few-shot beats SOTA in TriviaQA and is highly competitive in translation, it falls short on harder comprehension tasks.*

---

## Revision Summary

* **Paradigms:** Pre-train $\rightarrow$ Prompt $\rightarrow$ Predict replaced Pre-train $\rightarrow$ Fine-tune.
* **Prompt Types:** Zero-shot (0 examples), One-shot (1 example), Few-shot (limited by context window max).
* **Model Sensitivity:** LLMs are highly brittle to discrete (hard) prompt phrasing.
* **PEFT:** Freezes the LLM to save compute. **Prompt Tuning** prepends learnable tokens to the input; **Prefix Tuning** prepends to all layers.
* **Scaling Equivalency:** At massive parameter scales, Prompt Tuning achieves the same accuracy as full model fine-tuning.
* **Length & Init:** 20-30 tokens is the optimal soft prompt length. Random initialization works fine for massive models.

---